### Домашнее задание 2 (часть 1): дообучение BERT

В этом задании вы получите эмбеддинги с помощью BERT, дообучите предобученную модель на новых данных и сравните возможности трансформеров с предыдущими архитектурами, учитывая их более высокие вычислительные затраты.

**Всего: 10 баллов (2 + 5 + 3); все части обязательны.** Сдайте только заполненный ноутбук с сохранёнными результатами оценки и сравнения моделей. Баллы за код второй части линейно масштабируются по accuracy на полной валидационной выборке: 0 при 90% и ниже, все заработанные баллы при 92% и выше. Веса моделей и файлы эмбеддингов сдавать не нужно.

Заполните четыре ячейки TODO, выполните проверки и сохраните результаты оценки и сравнения в сдаваемом ноутбуке.


In [ ]:
%pip install --quiet "transformers==4.57.6" "datasets==4.5.0" "accelerate==1.12.0" "sentencepiece==0.2.1" "protobuf==5.29.5" "scikit-learn>=1.5,<2"

import os
os.environ["WANDB_DISABLED"] = "true"

import torch
import transformers
import datasets
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments


### Загрузка данных и модели


В этом задании используется **Quora Question Pairs (QQP)**.

Датасет содержит более 400 000 пар вопросов. Для каждой пары указано, являются ли вопросы перефразировками друг друга: речь идёт о дубликатах по смыслу, а не просто о связанных темах. Подробнее о данных можно прочитать [здесь](https://paperswithcode.com/dataset/quora-question-pairs).
Метки: 0 — не дубликаты, 1 — дубликаты.


In [ ]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Пример [0]:", qqp['train'][0])
print("Пример [3]:", qqp['train'][3])


In [ ]:
# Открытые проверки
for split in ['train', 'validation']:
    assert len(qqp[split]) > 0, f'{split}: датасет пуст.'
    assert set(qqp[split]['label']) == {0, 1}, f'{split}: ожидаются метки классов 0 и 1.'
    assert {'text1', 'text2', 'label'}.issubset(qqp[split].column_names), 'Сохраните оба вопроса и их метку.'


In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)


### Токенизация данных


Библиотека [datasets](https://huggingface.co/docs/datasets/en/index) позволяет применять функцию к данным с помощью `map`, как в функциональном программировании.

Что происходит с текстами в `qqp_preprocessed`?

- Исходные `text1` и `text2` преобразуются в числовые идентификаторы токенов с помощью соответствующего токенизатора.
- Для BERT тексты объединяются с разделителем `SEP`, а в начало добавляется токен `CLS`. Последовательность обрезается, если её длина превышает 128 токенов, или дополняется до 128 токенов, если она короче.
- Датасет `qqp_preprocessed` содержит:
    - _Идентификаторы токенов_ (`input_ids`): последовательность числовых идентификаторов.
    - _Маски внимания_ (`attention_mask`): бинарные маски, обозначающие позиции дополнения.
    - _Идентификаторы типов токенов_ (`token_type_ids`): различают токены из `text1` и `text2`.

__Обратите внимание!__ Маски внимания исключают позиции дополнения из внимания, но не отменяют все вычисления на этих позициях. `token_type_ids` могут отсутствовать.


In [ ]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)


In [ ]:
# Открытые проверки
assert model.config.num_labels == 2, 'Ожидаются два логита классов.'
for split in ['train', 'validation']:
    assert len(qqp_preprocessed[split]) == len(qqp[split]), 'Токенизация должна сохранять число строк.'
    np.testing.assert_array_equal(qqp_preprocessed[split]['label'], qqp[split]['label'], err_msg='Сохраните метки и порядок строк.')
    sample = qqp_preprocessed[split][0]
    assert len(sample['input_ids']) == len(sample['attention_mask']) == MAX_LENGTH, 'Дополните или обрежьте последовательность до MAX_LENGTH.'
    assert set(sample['attention_mask']).issubset({0, 1}), 'Маска внимания должна содержать только 0 и 1.'
    assert 0 <= min(sample['input_ids']) <= max(sample['input_ids']) < model.config.vocab_size, 'Идентификаторы токенов должны принадлежать словарю этой модели.'


In [ ]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")


### Оценка качества (2 балла)

Мы выбрали готовую модель, обученную на QQP. Насколько хорошо она работает?

Один из способов это проверить — вычислить accuracy на валидационной выборке. Далее вы реализуете эту оценку.

Ниже приведён код, с которого можно начать:


Посмотрим на наши данные.


In [ ]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)


In [ ]:
for batch in val_loader:
     break  # Посмотрим на один батч.
print("Пример батча:", batch)

model.eval()
with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nПредсказание (вероятности):', torch.softmax(predicted.logits, dim=1).data.numpy())


У модели одна классификационная голова с двумя выходными логитами — по одному на каждый класс.


__Ваша задача__ — вычислить accuracy модели на валидационной выборке.
Наивная реализация может работать несколько часов. Используйте следующие оптимизации:

- запускайте модель на GPU с `no_grad`;
- используйте размер батча больше 1;
- начните с `num_workers=0`; при необходимости измерьте скорость с дополнительными процессами загрузки данных;
- при желании используйте [смешанную точность](https://pytorch.org/docs/stable/notes/amp_examples.html).


Хотя вычисления модели выполняются на GPU, загрузка данных с диска или из памяти и преобразование в нужный модели формат, например в тензоры, выполняются на CPU.

Недостаточная производительность CPU может замедлять весь процесс.


In [ ]:
from tqdm import tqdm
import multiprocessing

cores = multiprocessing.cpu_count() # Число ядер процессора.
cores


In [ ]:
# Переносим модель на GPU, если он доступен.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Создаём DataLoader для валидационной выборки.
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16,  # Увеличенный батч ускоряет обработку.
    shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=0  # Подходит для Colab; дополнительные процессы загрузки данных — необязательная оптимизация.
)


In [ ]:
# Оценка accuracy на валидации.
# TODO 1: замените каждое ... кодом для обработки всех батчей и вычисления accuracy.
model.eval()
correct_predictions = 0
total_predictions = 0
use_amp = device.type == 'cuda'

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Оценка качества"):
        # Перенесите входы модели и метки на устройство; исключите служебные поля строк.
        inputs = ...
        labels = ...
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = ...
        # Получите предсказания классов и подсчитайте примеры, включая последний батч.
        ...

accuracy = ...  # Доля правильных ответов от 0 до 1.
print(f"Accuracy на валидации: {accuracy:.4f}")


In [ ]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy)
assert 0 <= accuracy <= 1
assert total_predictions == len(val_set), 'Обработайте каждый пример валидации ровно один раз.'
assert np.isclose(accuracy, correct_predictions / total_predictions)
baseline_accuracy = float(accuracy)
print('Опубликованная accuracy базовой модели на полной выборке — около 0.9084; другая длина или настройки могут изменить результат.')


### Дообучение модели (5 баллов)

В этой части вам нужно самостоятельно дообучить модель. Выберите предобученную модель __с архитектурой, отличной от оригинального BERT__, которую ранее не дообучали на QQP, в том числе в рамках GLUE или обучения на нескольких задачах. Рекомендуем [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Можно выбрать другую модель, опираясь на результаты публичных бенчмарков, например [GLUE](https://gluebenchmark.com/).

Вы можете написать цикл обучения самостоятельно или использовать `transformers.Trainer` (см. [пример](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Стремитесь к accuracy 92% на полной валидационной выборке. Оценка за вторую часть равна `C * clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1)`, где `C` — баллы за корректность кода, от 0 до 5. При accuracy 90% и ниже оценка равна 0; при 92% и выше вы получаете все заработанные за код баллы. Сравните результат с готовой моделью BERT из первой части. Оценивайте обе модели в одинаковых условиях, обучайтесь только на тренировочной выборке и сохраните вывод сравнения. Если вы остановили обучение раньше, оцените текущую модель.


In [ ]:
# TODO 2: выберите модель с архитектурой не BERT, ранее не дообученную на QQP, и токенизируйте пары.
# Освобождаем память GPU, занятую базовой моделью, перед загрузкой новой.
model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Загрузите выбранную модель и её токенизатор, например DeBERTa-v3.
model_name = ...
tokenizer = ...
model = ...  # Загрузите с num_labels=2 и dtype=torch.float32 для обучения с AMP.

# Повторно токенизируйте qqp новым токенизатором; можно использовать функцию предобработки выше.

# Определите функцию токенизации пар или используйте уже написанную.
...

# Токенизируйте qqp, сохранив метки и порядок строк в выборках.
qqp_preprocessed = ...


In [ ]:
# Открытые проверки
assert model.config.num_labels == 2, 'Используйте два логита для классификации на два класса.'
assert model.config.model_type != 'bert', 'Выберите архитектуру, отличную от оригинального BERT.'
for split in ['train', 'validation']:
    raw, encoded = qqp[split], qqp_preprocessed[split]
    assert len(encoded) == len(raw), 'Сохраните исходный размер выборки.'
    np.testing.assert_array_equal(encoded['label'], raw['label'], err_msg='Сохраните метки и порядок строк.')
    probe_ids = [0, len(raw) // 2, len(raw) - 1]
    raw_probe = raw.select(probe_ids)
    expected = tokenizer(list(raw_probe['text1']), list(raw_probe['text2']), padding='max_length',
                         truncation=True, max_length=MAX_LENGTH)
    np.testing.assert_array_equal(encoded.select(probe_ids)['input_ids'], expected['input_ids'],
                                  err_msg='Повторно токенизируйте оба исходных вопроса новым токенизатором.')
accuracy = None  # После прерванного обучения не используйте прежнюю accuracy модели BERT.


In [ ]:
# TODO 3: реализуйте метрику, настройте и запустите обучение, оцените качество.
# Подготавливаем тренировочную и валидационную выборки.
train_set = qqp_preprocessed['train']
val_set = qqp_preprocessed['validation']  


# Задаём метрику оценки качества. При желании напишите собственную реализацию.
from sklearn.metrics import accuracy_score

# При использовании transformers.Trainer можно воспользоваться функцией ниже.
def compute_metrics(eval_pred):
    """
    Вычисляет метрики при обучении или оценке модели.
    Аргументы:
        eval_pred (tuple): Кортеж, содержащий:
            - logits (ndarray или torch.Tensor): Логиты модели для каждого примера
              в батче оценки. Форма: (batch_size, num_classes).
            - labels (ndarray или torch.Tensor): Истинные метки каждого примера.
              Форма: (batch_size,).
    Возвращает:
        dict: Словарь вычисленных метрик:
            - "accuracy" (float): Доля правильных ответов среди всех примеров.
    """
    accuracy = ...  # Вычислите по eval_pred, не используя глобальную переменную accuracy.
    return {"accuracy": accuracy}

# Открытые проверки
metric_logits = np.array([[4, 1], [1, 4], [4, 1], [1, 4], [1, 4]])
metric_labels = np.array([0, 1, 0, 1, 0])
assert np.isclose(compute_metrics((metric_logits, metric_labels))['accuracy'], 0.8), 'Вычислите accuracy по примерам, выбрав предсказания вдоль измерения классов.'

# При желании напишите цикл обучения самостоятельно вместо transformers.Trainer.
# Начальное значение скорости обучения — 2e-5.
# При необходимости меняйте скорость обучения на порядок, например до 2e-4 или 2e-3.
# Трёх эпох обычно достаточно для дообучения без сильного забывания ранее изученного.
# Используйте регуляризацию weight_decay. Начните с 1e-2 и при необходимости измените значение.
# Можно выбирать лучшую модель по accuracy.

# Задайте параметры обучения без аргумента device: устройство выбирается автоматически.
# Оценивайте и сохраняйте модель каждые 1000 шагов оптимизатора; записывайте loss каждые 50 шагов.
training_args = TrainingArguments(
    ...
)

# Оставляем веса в FP32 для AMP и очищаем градиенты от предыдущего неудачного запуска.
model.float()
model.zero_grad(set_to_none=True)

# Классификатор возвращает средний loss батча; Trainer учитывает накопление градиентов.
model.accepts_loss_kwargs = False

# Создаём Trainer.
trainer = Trainer(
    ...
)

# Дообучаем модель.
...

# Оцените текущую модель на валидации и запишите результат в accuracy.
...
print(f"Accuracy на валидации: {accuracy:.4f}")


In [ ]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy), 'Оцените дообученную модель; не используйте accuracy базовой модели.'
assert 0 <= accuracy <= 1
if 'trainer' in globals():
    assert trainer.state.global_step > 0, 'Запустите обучение перед итоговой оценкой.'
    logged_losses = [record[key] for record in trainer.state.log_history for key in ('loss', 'train_loss') if key in record]
    assert all(np.isfinite(value) for value in logged_losses), 'Значения функции потерь при обучении должны быть конечными.'
import json
comparison = {
    'baseline_accuracy': float(baseline_accuracy),
    'finetuned_accuracy': float(accuracy),
    'delta': float(accuracy - baseline_accuracy),
    'validation_examples': len(qqp['validation']),
}
print(f'Базовая модель BERT: {baseline_accuracy:.6f}; ваша модель: {accuracy:.6f}; прирост: {accuracy - baseline_accuracy:+.6f}')
print('Результат базовой модели BERT превышен:', accuracy > baseline_accuracy)
accuracy_factor = float(np.clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1))
print(f'Коэффициент качества второй части: {accuracy_factor:.4f}; максимум баллов за вторую часть при этой accuracy: {5 * accuracy_factor:.4f}/5')
print('QQP_COMPARISON ' + json.dumps(comparison))


Используйте валидационную выборку для оценки качества и выбора модели. Метки тестовой выборки QQP скрыты (`-1`), поэтому вычислить accuracy на ней локально нельзя.


### Поиск с помощью эмбеддингов BERT (3 балла, обязательная часть)

Теперь попробуем использовать эмбеддинги на основе BERT для поиска информации.

Обычный BERT и похожие модели не обучаются напрямую с контрастивной функцией потерь или функцией потерь для троек примеров, которые сближают эмбеддинги похожих текстов. Поэтому для поисковой системы лучше выбрать специализированную [модель для оценки сходства предложений](https://huggingface.co/models?pipeline_tag=sentence-similarity). Выберите модель, подходящую для этой задачи.

Как и в первом домашнем задании, вам нужно построить поисковую систему:

1. _Подготовьте базу эмбеддингов._ Из пар вопросов QQP возьмите только поле `text1` валидационной выборки (`validation`). Получите эмбеддинги выбранной моделью и сохраните их в `numpy.ndarray` для последующего использования. При желании можно использовать индекс [Faiss](https://github.com/facebookresearch/faiss).
2. _Реализуйте поиск вопросов, похожих на запрос._ Напишите функцию или класс для работы с базой. __Результат этой части оценивается по способности понятно вывести пять наиболее похожих вопросов Quora для произвольного нового запроса.__

Сравните, насколько лучше такой поиск учитывает смысл текстов по сравнению с предыдущим подходом.

Помимо вывода, возвращайте `(row_ids, cosine_scores)` по убыванию косинусной близости. Число результатов равно `min(top_k, len(database))`. Индексы строк базы должны соответствовать порядку исходных `text1`.


In [ ]:
# TODO 4: выберите энкодер, создайте database и реализуйте поиск по запросу.
# Инициализируем модель и её токенизатор.
model_name = ...
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

all_texts = list(qqp['validation']['text1'])
# Получите по одной строке эмбеддингов NumPy для каждого исходного text1 в том же порядке.
# Используйте этот токенизатор и учитывайте дополнение при усреднении эмбеддингов токенов.
database = ...

def find_similar_questions(query, database, model, top_k = 5):
    """
    Находит и выводит top_k вопросов, наиболее похожих на запрос.

    Кодирует запрос, сравнивает его с заранее вычисленной базой
    эмбеддингов по косинусной близости и выводит наиболее
    похожие по смыслу вопросы.

    Аргументы:
        query (str): Поисковый запрос пользователя.
        database (np.ndarray): Двумерный массив эмбеддингов вопросов в базе.
        model: Модель, используемая со своим токенизатором для кодирования запроса.
        top_k (int): Число результатов для вывода.

    Возвращает:
        tuple: (row_ids, cosine_scores) по убыванию близости, помимо вывода.
    """
    ...


In [ ]:
# Открытые проверки
assert isinstance(database, np.ndarray) and database.ndim == 2
assert database.shape[0] == len(all_texts) == len(qqp['validation'])
assert np.isfinite(database).all() and (np.linalg.norm(database, axis=1) > 0).all()
row_ids, cosine_scores = find_similar_questions("Are apples useful?", database, model, top_k=5)
row_ids, cosine_scores = np.asarray(row_ids), np.asarray(cosine_scores)
assert row_ids.ndim == cosine_scores.ndim == 1
assert len(row_ids) == len(cosine_scores) == min(5, len(database))
assert np.issubdtype(row_ids.dtype, np.integer) and len(set(row_ids.tolist())) == len(row_ids)
assert ((row_ids >= 0) & (row_ids < len(database))).all()
assert np.isfinite(cosine_scores).all() and (np.abs(cosine_scores) <= 1 + 2e-3).all()
assert (np.diff(cosine_scores) <= 1e-6).all(), 'Возвращайте результаты по убыванию косинусной близости.'
